# 9-4. Polar码与Turbo码

**Polar Codes and Turbo Codes**

---

本notebook探讨现代编码理论中的两种重要技术：Polar码和Turbo码。Polar码是5G NR控制信道的编码方案，基于信道极化理论；Turbo码是3G/4G数据信道的编码方案，性能接近香农极限。

我们还将介绍BICM（比特交织编码调制），这是现代通信系统中广泛使用的调制编码联合设计方案。

## 1. 学习目标

通过本notebook，您将：

- **理解极化理论**：信道合并、分裂与极化现象
- **掌握PB构造方法**：极化权重（Polarization Weight）构造原理
- **理解SC译码**：串行抵消译码的原理与实现
- **理解SCL-CRC译码**：CRC辅助的列表译码及性能增益
- **理解Turbo码**：并行级联卷积码的编码与迭代译码
- **理解BICM概念**：比特交织编码调制的优势与应用

## 2. 极化码理论

### 2.1 信道极化原理

极化码的核心思想是对 $N$ 个独立信道 $W$ 进行信道合并与分裂，构造出 $N$ 个极化信道。其中一部分信道变得"完美"（容量为1），另一部分变得"糟糕"（容量接近0）。

**信道合并**：将 $N=2^n$ 个独立信道 $W$ 合并为一个复合信道 $W_N$。

对于 $n=1$（两个信道合并）：

$$W_2(y_1, y_2 | u_1, u_2) = W(y_1 | u_1 \oplus u_2) W(y_2 | u_2)$$

**信道分裂**：从复合信道 $W_N$ 分裂出 $N$ 个极化信道 $W_N^{(i)}$。

$$W_N^{(i)}(y_1^N, u_1^{i-1} | u_i) = \sum_{u_{i+1}^N} \frac{1}{2^{N-1}} W_N(y_1^N | u_1^N)$$

### 2.2 极化权重（PB）构造

极化权重用于确定信息比特和冻结比特的位置。 Arikan提出的原始构造基于密度进化，但计算复杂。

**PB（Polarization Weight）构造方法**：

$$PB(i) = \sum_{j=0}^{n-1} \lfloor \frac{i}{2^j} \rfloor \mod 2 \cdot \frac{1}{4^{j+1}}$$

其中 $i$ 是信道索引（$0 \leq i < N$），$n = \log_2 N$。

**构造步骤**：
1. 计算所有信道的PB值
2. 按PB值从小到大排序
3. 前 $K$ 个信道分配信息比特（可靠性高）
4. 后 $N-K$ 个信道分配冻结比特（设为已知值，通常为0）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('环境配置完成')

### 2.3 极化权重计算实现

In [ ]:
def polarization_weight(i, n):
    """
    计算极化权重（Polarization Weight）
    
    参数:
        i: 信道索引 [0, N)
        n: log2(N)，极化阶数
    
    返回:
        PB(i): 极化权重值
    """
    pb = 0.0
    for j in range(n):
        # 取i的第j位（从低位开始）
        bit = (i >> j) & 1
        pb += bit * (1.0 / (4 ** (j + 1)))
    return pb

def generate_pb_sequence(N):
    """
    生成极化权重序列并排序
    
    参数:
        N: 信道总数（2的幂）
    
    返回:
        sorted_indices: 按PB值排序的信道索引
    """
    n = int(np.log2(N))
    pb_values = [polarization_weight(i, n) for i in range(N)]
    # 按PB值排序，返回索引
    sorted_indices = np.argsort(pb_values)
    return sorted_indices, pb_values

# 测试：N=8 极化码
N = 8
sorted_idx, pb_vals = generate_pb_sequence(N)

print(f'N={N} 极化码的PB序列:')
print(f'原始PB值: {[f"{v:.4f}" for v in pb_vals]}')
print(f'排序索引: {sorted_idx}')
print(f'信息比特位置: {sorted_idx[-2:]} (可靠性最高)')
print(f'冻结比特位置: {sorted_idx[:6]} (可靠性最低)')

### 2.4 可视化极化权重分布

In [ ]:
def plot_pb_distribution(N_list=[8, 16, 32, 64]):
    """绘制不同N值下PB值的分布"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    for idx, N in enumerate(N_list):
        sorted_idx, pb_vals = generate_pb_sequence(N)
        
        ax = axes[idx]
        # 绘制PB值分布
        ax.bar(range(N), pb_vals, color='steelblue', alpha=0.7)
        ax.set_xlabel('信道索引 i')
        ax.set_ylabel('极化权重 PB(i)')
        ax.set_title(f'N={N} 极化码 PB分布')
        ax.grid(True, alpha=0.3)
        
        # 标记信息比特位置（假设K=N/2）
        K = N // 2
        info_pos = sorted_idx[-K:]
        freeze_pos = sorted_idx[:N-K]
        
        for pos in info_pos:
            ax.axvline(x=pos, color='red', linestyle='--', alpha=0.5, linewidth=0.5)
        
    plt.tight_layout()
    plt.savefig('pb_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('PB分布图已保存')}

plot_pb_distribution()

## 3. Polar码编码

### 3.1 编码原理

Polar码通过极化变换矩阵 $G_N = B_N F^{\otimes n}$ 编码，其中：

- $F = \begin{bmatrix} 1 & 0 \\ 1 & 1 \end{bmatrix}$ 是基础变换矩阵
- $F^{\otimes n}$ 是 $F$ 的 $n$ 次Kronecker积
- $B_N$ 是位反转（bit-reversal）排列矩阵

**编码公式**：

$$x_1^N = u_1^N G_N$$

其中 $u_1^N$ 是输入比特向量，$x_1^N$ 是编码后比特。

In [ ]:
def kron_product(mat_list):
    """计算多个矩阵的Kronecker积"""
    result = mat_list[0]
    for mat in mat_list[1:]:
        result = np.kron(result, mat)
    return result

def polar_generator_matrix(N):
    """
    生成极化码生成矩阵 G_N = B_N * F^{\otimes n}
    
    参数:
        N: 码长（2的幂）
    
    返回:
        G_N: N x N 的生成矩阵
    """
    n = int(np.log2(N))
    F = np.array([[1, 0], [1, 1]], dtype=np.int32)
    
    # 计算F^{\otimes n}
    F_power = F
    for _ in range(n - 1):
        F_power = np.kron(F_power, F)
    
    # 位反转排列矩阵
    bit_reversal = np.zeros((N, N), dtype=np.int32)
    for i in range(N):
        # 反转i的二进制位
        reversed_i = int(format(i, f'0{n}b')[::-1], 2)
        bit_reversal[i, reversed_i] = 1
    
    # G_N = B_N * F^{\otimes n}
    G_N = bit_reversal @ F_power
    return G_N

# 测试：生成N=8的极化码生成矩阵
N = 8
G = polar_generator_matrix(N)
print(f'N={N} 极化码生成矩阵 G_{N}:')
print(G)

In [ ]:
def polar_encode(u, N):
    """
    Polar码编码
    
    参数:
        u: 输入比特向量（长度N）
        N: 码长
    
    返回:
        x: 编码比特向量
    """
    G = polar_generator_matrix(N)
    u = np.array(u, dtype=np.int32)
    # 模2乘法
    x = (G @ u) % 2
    return x

# 示例：编码消息
N = 8
# 假设信息比特位置（根据可靠性选择）
message = [1, 0, 1]  # K=3个信息比特
K = len(message)

sorted_idx, _ = generate_pb_sequence(N)
# 构建完整输入向量
u = np.zeros(N, dtype=np.int32)
info_positions = sorted_idx[-K:]  # 可靠性最高的位置
u[info_positions] = message

x = polar_encode(u, N)
print(f'信息比特: {message}')
print(f'信息位置: {info_positions}')
print(f'完整输入u: {u}')
print(f'编码输出x: {x}')

## 4. SC译码器

### 4.1 SC译码原理

SC（Successive Cancellation）译码按顺序估计每个比特 $u_i$：

1. 利用接收信号 $y$ 和已译出的前 $i-1$ 个比特 $\hat{u}_1^{i-1}$
2. 计算似然比（Log-Likelihood Ratio, LLR）：

$$LLR_i = \ln \frac{W_N^{(i)}(y, \hat{u}_1^{i-1} | u_i=0)}{W_N^{(i)}(y, \hat{u}_1^{i-1} | u_i=1)}$$

3. 判决规则：
   - $LLR_i \geq 0 \Rightarrow \hat{u}_i = 0$
   - $LLR_i < 0 \Rightarrow \hat{u}_i = 1$
   - 对于冻结比特，直接设 $\hat{u}_i = 0$

In [ ]:
def bhattacharyya_bound(W, y=None):
    """
    计算Bhattacharyya参数（信道可靠性度量）
    对于BEC(擦除信道)，Z(W) = sqrt(epsilon)
    
    参数:
        W: 信道参数（擦除概率对于BEC）
        y: 接收信号（可选）
    
    返回:
        Z: Bhattacharyya参数
    """
    return np.sqrt(W * (1 - W))

def bec_channel(epsilon, snr_db=None):
    """
    创建BEC（Binary Erasure Channel）模型
    
    参数:
        epsilon: 擦除概率
        snr_db: 信噪比（dB），如果提供则考虑噪声
    
    返回:
        channel_params: 信道参数字典
    """
    Z = bhattacharyya_bound(0, epsilon)
    return {
        'type': 'BEC',
        'epsilon': epsilon,
        'Z': Z,
        'capacity': 1 - epsilon
    }

# 测试：BEC信道
epsilon = 0.5  # 50%擦除概率
channel = bec_channel(epsilon)
print(f'BEC信道 (epsilon={epsilon}):')
print(f'  容量: {channel["capacity"]:.4f}')
print(f'  Bhattacharyya参数: {channel["Z"]:.4f}')

In [ ]:
def sc_decode(y, frozen_bits, N, channel_params):
    """
    SC（Successive Cancellation）译码
    
    参数:
        y: 接收信号（对数似然比形式）
        frozen_bits: 冻结比特位置的集合
        N: 码长
        channel_params: 信道参数
    
    返回:
        u_hat: 译码后的比特向量
    """
    n = int(np.log2(N))
    u_hat = np.zeros(N, dtype=np.int32)
    
    # 简化的SC译码：基于信道参数和冻结比特位置
    # 在实际实现中，需要递归计算LLR
    for i in range(N):
        if i in frozen_bits:
            u_hat[i] = 0  # 冻结比特设为0
        else:
            # 计算似然比（简化版本）
            llr = y[i] if i < len(y) else 0
            
            # 根据极化信道可靠性调整LLR
            # 这里使用简化的模型
            sorted_idx, _ = generate_pb_sequence(N)
            reliability_rank = np.where(sorted_idx == i)[0][0]
            
            # 信道越可靠，LLR越准确
            llr_adjusted = llr * (reliability_rank / N)
            
            u_hat[i] = 0 if llr_adjusted >= 0 else 1
    
    return u_hat

# 示例：SC译码
N = 8
sorted_idx, _ = generate_pb_sequence(N)
frozen_bits = set(sorted_idx[:N-3])  # 冻结位置（可靠性最低）

# 模拟接收信号（LLR形式）
y = np.random.randn(N) * 2  # 假设高SNR

u_decoded = sc_decode(y, frozen_bits, N, channel)
print(f'冻结比特位置: {sorted(sorted_idx[:N-3])}')
print(f'译码结果: {u_decoded}')

## 5. SCL-CRC译码器

### 5.1 列表译码原理

SCL（Successive Cancellation List）译码在SC基础上维护 $L$ 个候选路径：

1. 每个比特位置尝试两种可能性（0和1）
2. 使用路径度量（Path Metric, PM）评估每个路径
3. 选择PM最好的 $L$ 条路径继续

$$PM^{(l)} = \sum_{i=1}^{N} \ln(1 + e^{-|LLR_i^{(l)}|})$$

### 5.2 CRC辅助译码

在SCL基础上添加CRC校验：

1. 对每条候选路径计算CRC校验
2. 通过CRC检验的路径才是有效路径
3. 如果有多条通过CRC，选择PM最大的那条

CRC可以显著提升极化码的性能，在5G NR中被广泛使用。

In [ ]:
def crc_encode(bits, poly=0x1021):
    """
    CRC编码（使用CRC-16-MCC标准的简化版本）
    
    参数:
        bits: 输入比特数组
        poly: 生成多项式
    
    返回:
        crc: CRC校验位
    """
    crc = 0xFFFF
    for bit in bits:
        crc ^= (bit << 15)
        for _ in range(16):
            if crc & 0x8000:
                crc = ((crc << 1) ^ poly) & 0xFFFF
            else:
                crc = (crc << 1) & 0xFFFF
    return (~crc) & 0xFFFF

def crc_check(bits, crc_value, poly=0x1021):
    """
    CRC校验
    
    参数:
        bits: 接收比特数组
        crc_value: 接收的CRC值
        poly: 生成多项式
    
    返回:
        True如果CRC校验通过
    """
    computed_crc = crc_encode(bits, poly)
    return computed_crc == crc_value

# 测试CRC
test_bits = [1, 0, 1, 1, 0, 0, 1]
crc_val = crc_encode(test_bits)
print(f'测试比特: {test_bits}')
print(f'CRC值: {hex(crc_val)}')
print(f'CRC校验: {crc_check(test_bits, crc_val)}')

In [ ]:
def scl_crc_decode(y, frozen_bits, info_positions, N, L=8, crc_bits=16):
    """
    SCL-CRC译码器
    
    参数:
        y: 接收信号（LLR）
        frozen_bits: 冻结比特位置集合
        info_positions: 信息比特位置集合
        N: 码长
        L: 列表大小
        crc_bits: CRC校验位长度
    
    返回:
        decoded_message: 译码后的信息比特
    """
    # 初始化路径列表
    # 每条路径: {'bits': u向量, 'pm': 路径度量}
    paths = [{'bits': np.zeros(N, dtype=np.int32), 'pm': 0.0}]
    
    for i in range(N):
        new_paths = []
        
        if i in frozen_bits:
            # 冻结比特：只保留一条路径
            for path in paths:
                path['bits'][i] = 0
            new_paths = paths
        else:
            # 信息比特或CRC比特：扩展所有路径
            for path in paths:
                # 尝试比特0
                new_path_0 = path.copy()
                new_path_0['bits'] = path['bits'].copy()
                new_path_0['bits'][i] = 0
                llr = y[i] if i < len(y) else 1.0
                new_path_0['pm'] += np.log(1 + np.exp(-abs(llr)))
                new_paths.append(new_path_0)
                
                # 尝试比特1
                new_path_1 = path.copy()
                new_path_1['bits'] = path['bits'].copy()
                new_path_1['bits'][i] = 1
                new_path_1['pm'] += np.log(1 + np.exp(-abs(llr)))
                new_paths.append(new_path_1)
        
        # 按PM排序，保留最好的L条路径
        new_paths.sort(key=lambda x: x['pm'])
        paths = new_paths[:L]
    
    # CRC校验
    info_bits_len = len(info_positions) - crc_bits
    for path in paths:
        info_bits = [path['bits'][pos] for pos in info_positions[:info_bits_len]]
        crc_value = 0
        for pos in info_positions[info_bits_len:]:
            crc_value = (crc_value << 1) | path['bits'][pos]
        
        if crc_check(info_bits, crc_value):
            return info_bits, path['bits']
    
    # 没有通过CRC的路径，返回PM最小的
    best_path = min(paths, key=lambda x: x['pm'])
    info_bits = [best_path['bits'][pos] for pos in info_positions[:info_bits_len]]
    return info_bits, best_path['bits']

# 示例：SCL-CRC译码
N = 16
sorted_idx, _ = generate_pb_sequence(N)
K = 8
info_positions = list(sorted_idx[-K:])
frozen_bits = set(sorted_idx[:N-K])

# 模拟接收信号
y = np.random.randn(N) * 3

decoded_msg, decoded_full = scl_crc_decode(y, frozen_bits, info_positions, N, L=8)
print(f'信息比特位置: {info_positions}')
print(f'译码信息比特: {decoded_msg}')

## 6. Turbo码

### 6.1 并行级联卷积码

Turbo码由两个递归系统卷积码（RSC）并行级联构成：

```
           ┌─────────┐
     u ────┤  RSC1   ├─── y1 (校验位1)
           └───┬─────┘
               │
           ┌───┴───┐
           │交织器 │
           └───┬───┘
               │
           ┌───┴─────┐
     u ────┤  RSC2   ├─── y2 (校验位2)
           └───────────┘
```

**关键组件**：
- **RSC编码器**：递归系统卷积码，有反馈
- **交织器**：将信息比特重排，打散相关性
- **删除器**：可选，删除部分校验位以调整码率

In [ ]:
def rsc_encode(bits, gen_poly=[0b111, 0b101], feedback=0b100):
    """
    递归系统卷积码（RSC）编码器
    
    参数:
        bits: 输入信息比特
        gen_poly: 生成多项式 [g1, g2]，g1是系统位
        feedback: 反馈连接
    
    返回:
        coded_bits: 编码后的比特（系统位 + 校验位）
    """
    n = len(bits)
    k = 3  # 约束长度
    
    # 移位寄存器状态
    state = 0
    
    system_bits = []  # 系统位（输出）
    parity_bits = []  # 校验位
    
    for bit in bits:
        # 计算输出
        # 系统位直接输出
        system_bits.append(bit)
        
        # 校验位 = 输入位 * g1 + 状态 * g2 + 反馈
        # 简化的计算
        parity = (bit ^ (state & 1)) & 1
        parity_bits.append(parity)
        
        # 更新状态（移位）
        new_bit = (bit ^ ((state >> 1) & 1)) & 1
        state = ((state >> 1) | (new_bit << (k - 2))) & ((1 << (k - 1)) - 1)
    
    # 合并系统位和校验位
    coded_bits = []
    for s, p in zip(system_bits, parity_bits):
        coded_bits.append(s)
        coded_bits.append(p)
    
    return coded_bits

def turbo_encode(bits, gen_poly=[0b111, 0b101], interleave_depth=None):
    """
    Turbo码编码器
    
    参数:
        bits: 输入信息比特
        gen_poly: 生成多项式
        interleave_depth: 交织深度，默认使用len(bits)
    
    返回:
        turbo_codeword: Turbo编码后的比特
    """
    if interleave_depth is None:
        interleave_depth = len(bits)
    
    # 第一个RSC编码器
    coded1 = rsc_encode(bits, gen_poly)
    
    # 交织
    np.random.seed(42)  # 固定随机种子以保证可重复
    interleaved_idx = np.random.permutation(interleave_depth) % len(bits)
    interleaved_bits = [bits[i] for i in interleaved_idx]
    
    # 第二个RSC编码器
    coded2 = rsc_encode(interleaved_bits, gen_poly)
    
    # 组合（系统位 + 校验位1 + 校验位2）
    # 简化：交替添加校验位
    turbo_codeword = []
    for i in range(len(bits)):
        # 系统位
        turbo_codeword.append(bits[i])
        # 第一个编码器的校验位
        if 2*i + 1 < len(coded1):
            turbo_codeword.append(coded1[2*i + 1])
        # 第二个编码器的校验位
        if 2*i + 1 < len(coded2):
            turbo_codeword.append(coded2[2*i + 1])
    
    return turbo_codeword

# 测试Turbo编码
message = [1, 0, 1, 1, 0, 1, 0, 0]
turbo_code = turbo_encode(message)
print(f'原始信息: {message}')
print(f'Turbo编码: {turbo_code}')
print(f'码率: {len(message)/len(turbo_code):.2f}')

### 6.2 迭代译码

Turbo码使用迭代译码（Turbo迭代）：

1. **分量译码器1**：使用接收信号和先验信息，输出外信息
2. **解交织**：将外信息重排
3. **分量译码器2**：使用交织后的接收信号和先验信息
4. **交织**：将外信息重排回原来的顺序
5. 重复步骤1-4直到收敛或达到最大迭代次数

**软输出译码器（SISO）**使用BCJR算法计算软信息。

In [ ]:
def bcjr_decode(llr, gen_poly=[0b111, 0b101]):
    """
    BCJR算法软输入软输出译码
    简化的BCJR实现
    
    参数:
        llr: 接收信号的对数似然比
        gen_poly: 生成多项式
    
    返回:
        extrinsic: 外信息（软输出）
    """
    n = len(llr) // 2  # 系统位+校验位
    k = 3  # 约束长度
    
    # 简化实现：使用硬判决 + 小幅扰动
    extrinsic = np.zeros(n)
    
    for i in range(n):
        # 硬判决
        bit_estimate = 1 if llr[2*i] < 0 else 0
        
        # 简化的外信息计算
        parity_llr = llr[2*i + 1] if 2*i + 1 < len(llr) else 0
        
        # 外信息 = 系统位LLR + 校验位LLR的一部分
        extrinsic[i] = 0.5 * parity_llr * (1 if bit_estimate == 0 else -1)
    
    return extrinsic

def turbo_decode(received_llr, max_iterations=5):
    """
    Turbo码迭代译码
    
    参数:
        received_llr: 接收信号的对数似然比
        max_iterations: 最大迭代次数
    
    返回:
        decoded_bits: 译码后的比特
        iterations: 实际迭代次数
    """
    n = len(received_llr) // 3
    
    # 先验信息（初始化为0）
    prior1 = np.zeros(n)
    prior2 = np.zeros(n)
    
    decoded_bits = np.zeros(n, dtype=np.int32)
    
    for iteration in range(max_iterations):
        # 分量译码器1
        extrinsic1 = bcjr_decode(received_llr + np.concatenate([prior1, prior1, prior1]))
        
        # 解交织（简化的交织索引）
        np.random.seed(42)
        deinterleave_idx = np.argsort(np.random.permutation(n))
        interleaved_prior2 = prior1[deinterleave_idx]
        
        # 分量译码器2
        extrinsic2 = bcjr_decode(received_llr + np.concatenate([interleaved_prior2, interleaved_prior2, interleaved_prior2]))
        
        # 解交织得到新的先验信息
        prior1 = extrinsic2[deinterleave_idx]
        prior2 = extrinsic1
        
        # 更新译码结果
        total_llr = prior2 + extrinsic1
        decoded_bits = (total_llr < 0).astype(np.int32)
    
    return decoded_bits, iteration + 1

# 示例：Turbo译码
message = [1, 0, 1, 1, 0, 1, 0, 0]
turbo_code = turbo_encode(message)

# 模拟信道：添加噪声得到LLR
snr_db = 3
noise_var = 10 ** (-snr_db / 10)
received = [1 if b == 0 else -1 for b in turbo_code]
received = [b + np.random.randn() *np.sqrt(noise_var) for b in received]
llr = np.array([-2*b/noise_var for b in received])  # 转换为LLR

decoded, iters = turbo_decode(llr, max_iterations=5)
print(f'原始信息: {message}')
print(f'译码结果: {list(decoded)}')
print(f'迭代次数: {iters}')
print(f'译码正确: {list(decoded) == message}')

## 7. BICM（比特交织编码调制）

### 7.1 概念与优势

BICM（Bit-Interleaved Coded Modulation）将信道编码与高阶调制结合：

```
     信息 ──> 编码器 ──> 交织器 ──> 映射器 ──> 调制 ──> 信道
                                   ↑
                                   │
                              比特交织
```

**BICM的优势**：
- **频谱效率高**：结合高阶调制（如16-QAM、64-QAM）
- **灵活性强**：编码与调制独立设计
- **迭代检测**：可与迭代检测结合（BIT-ACM）

**核心思想**：比特交织打散错误，提高编码增益

In [ ]:
def gray_mapping(mod_order=4):
    """
    生成格雷码映射表
    
    参数:
        mod_order: 调制阶数（4=QPSK, 16=16-QAM, 64=64-QAM）
    
    返回:
        mapping: 比特到星座点的映射字典
    """
    m = int(np.log2(mod_order))
    mapping = {}
    
    # 生成格雷码序列
    for i in range(mod_order):
        gray_code = i ^ (i >> 1)
        # 计算I/Q分量
        if mod_order == 4:  # QPSK
            real = 1 if (gray_code & 1) == 0 else -1
            imag = 1 if (gray_code & 2) == 0 else -1
        elif mod_order == 16:  # 16-QAM
            # 4x4网格
            real = 2 * ((gray_code & 1) * 2 - 1)
            imag = 2 * (((gray_code >> 1) & 1) * 2 - 1)
            # 第二层格雷码
            real += (gray_code >> 2) - 2 if (gray_code >> 2) >= 2 else (gray_code >> 2)
            imag += (gray_code >> 3) - 2 if (gray_code >> 3) >= 2 else (gray_code >> 3)
        else:
            real = (i % 4) - 1.5
            imag = (i // 4) - 1.5
        
        # 转换为比特字符串
        bits = format(gray_code, f'0{m}b')
        mapping[bits] = complex(real, imag)
    
    return mapping

def bicm_transmitter(bits, mod_order=4, interleaver_seed=42):
    """
    BICM发射机
    
    参数:
        bits: 输入信息比特
        mod_order: 调制阶数
        interleaver_seed: 交织器随机种子
    
    返回:
        symbols: 发送的星座点符号
    """
    m = int(np.log2(mod_order))
    mapping = gray_mapping(mod_order)
    
    # 交织
    np.random.seed(interleaver_seed)
    n_symbols = len(bits) // m
    interleaved_bits = list(np.random.permutation(len(bits)))
    interleaved_bits = [bits[i] for i in interleaved_bits[:n_symbols*m]]
    
    # 映射
    symbols = []
    for i in range(n_symbols):
        bit_group = interleaved_bits[i*m:(i+1)*m]
        bit_str = ''.join(map(str, bit_group))
        if bit_str in mapping:
            symbols.append(mapping[bit_str])
        else:
            symbols.append(complex(0, 0))
    
    return symbols

def plot_constellation(mod_order=4):
    """绘制星座图"""
    mapping = gray_mapping(mod_order)
    
    plt.figure(figsize=(6, 6))
    for bits, symbol in mapping.items():
        plt.plot(symbol.real, symbol.imag, 'bo', markersize=10)
        plt.text(symbol.real + 0.1, symbol.imag + 0.1, bits,
                fontsize=10, ha='left', va='bottom')
    
    plt.grid(True)
    plt.axhline(y=0, color='k', linewidth=0.5)
    plt.axvline(x=0, color='k', linewidth=0.5)
    plt.xlabel('I (同相)')
    plt.ylabel('Q (正交)')
    plt.title(f'格雷映射 {mod_order}-QAM 星座图')
    plt.axis('equal')
    plt.xlim(-2.5, 2.5)
    plt.ylim(-2.5, 2.5)
    plt.savefig(f'constellation_{mod_order}qam.png', dpi=150, bbox_inches='tight')
    plt.show()

print('格雷映射QPSK星座图:')
plot_constellation(4)

## 8. 综合演示：Polar-Turbo系统

In [ ]:
def awgn_channel(symbols, snr_db):
    """
    AWGN信道
    
    参数:
        symbols: 发送符号
        snr_db: 信噪比（dB）
    
    返回:
        received: 接收符号
    """
    snr_linear = 10 ** (snr_db / 10)
    # 假设每个符号能量为1
    noise_var = 1.0 / snr_linear
    noise = np.random.randn(len(symbols)) * np.sqrt(noise_var / 2) + \
            1j * np.random.randn(len(symbols)) * np.sqrt(noise_var / 2)
    received = np.array(symbols) + noise
    return received

def simulate_polar_system(N=16, K=8, snr_db=3, n_trials=100):
    """
    仿真Polar码系统
    
    参数:
        N: 码长
        K: 信息比特数
        snr_db: 信噪比
        n_trials: 仿真次数
    
    返回:
        fer: 帧错误率
    """
    sorted_idx, _ = generate_pb_sequence(N)
    info_positions = list(sorted_idx[-K:])
    frozen_bits = set(sorted_idx[:N-K])
    
    errors = 0
    for _ in range(n_trials):
        # 生成随机信息
        message = np.random.randint(0, 2, K)
        
        # Polar编码
        u = np.zeros(N, dtype=np.int32)
        u[info_positions] = message
        codeword = polar_encode(u, N)
        
        # BPSK调制 + AWGN信道
        symbols = [1 - 2 * b for b in codeword]
        received = awgn_channel(symbols, snr_db)
        
        # 计算LLR
        llr = 2 * np.real(received) / (0.1 + 10**(-snr_db/10))
        
        # SC译码
        decoded = sc_decode(llr, frozen_bits, N, None)
        decoded_message = [decoded[pos] for pos in info_positions]
        
        if not np.array_equal(decoded_message, message):
            errors += 1
    
    return errors / n_trials

# 仿真不同SNR下的Polar码性能
print('Polar码SC译码性能仿真...')
snr_range = [0, 2, 4, 6, 8]
fer_results = []

for snr in snr_range:
    fer = simulate_polar_system(N=32, K=16, snr_db=snr, n_trials=50)
    fer_results.append(fer)
    print(f'  SNR={snr}dB: FER={fer:.4f}')

plt.figure(figsize=(10, 6))
plt.semilogy(snr_range, [max(f, 1e-6) for f in fer_results], 'b-o', linewidth=2)
plt.xlabel('SNR (dB)')
plt.ylabel('帧错误率 (FER)')
plt.title('Polar码 SC译码性能')
plt.grid(True, which='both')
plt.savefig('polar_performance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def simulate_turbo_system(info_len=8, snr_db=3, n_trials=100):
    """
    仿真Turbo码系统
    
    参数:
        info_len: 信息比特长度
        snr_db: 信噪比
        n_trials: 仿真次数
    
    返回:
        ber: 比特错误率
    """
    errors = 0
    total_bits = info_len * n_trials
    
    for _ in range(n_trials):
        # 生成随机信息
        message = list(np.random.randint(0, 2, info_len))
        
        # Turbo编码
        codeword = turbo_encode(message)
        
        # BPSK调制 + AWGN
        symbols = [1 - 2 * b for b in codeword]
        noise_var = 10 ** (-snr_db / 10)
        received = [s + np.random.randn() * np.sqrt(noise_var) for s in symbols]
        
        # 计算LLR
        llr = np.array([-2 * r / noise_var for r in received])
        
        # Turbo译码
        decoded, _ = turbo_decode(llr, max_iterations=5)
        
        errors += np.sum(decoded != message)
    
    return errors / total_bits

# 仿真Turbo码性能
print('Turbo码迭代译码性能仿真...')
snr_range = [0, 1, 2, 3, 4]
ber_results = []

for snr in snr_range:
    ber = simulate_turbo_system(info_len=8, snr_db=snr, n_trials=50)
    ber_results.append(ber)
    print(f'  SNR={snr}dB: BER={ber:.4f}')

plt.figure(figsize=(10, 6))
plt.semilogy(snr_range, [max(b, 1e-6) for b in ber_results], 'r-s', linewidth=2)
plt.xlabel('SNR (dB)')
plt.ylabel('比特错误率 (BER)')
plt.title('Turbo码迭代译码性能')
plt.grid(True, which='both')
plt.savefig('turbo_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. 总结与思考题

### 9.1 核心概念总结

| 概念 | 描述 |
|------|------|
| 信道极化 | 通过信道合并与分裂，使一部分信道容量趋近1，另一部分趋近0 |
| PB构造 | 基于信道可靠性的排序，确定信息比特和冻结比特位置 |
| SC译码 | 串行抵消，按顺序估计每个比特 |
| SCL-CRC | 列表译码 + CRC校验，提升极化码性能 |
| Turbo码 | 并行级联两个RSC编码器，使用迭代译码 |
| BICM | 比特交织编码调制，结合高阶调制提升频谱效率 |

### 9.2 思考题

**基础理解**：
1. Polar码为什么需要"冻结比特"？冻结比特的位置如何确定？
2. 在SC译码中，如果前面的比特译码错误，会如何影响后续比特的译码？

**深入分析**：
3. Turbo码的交织器设计对性能有什么影响？为什么交织可以提高编码增益？
4. 比较SC和SCL-CRC译码，分析CRC如何提升Polar码的纠错能力？

**工程实践**：
5. 在5G NR中，Polar码和LDPC码分别用于哪些场景？为什么？
6. BICM中的比特级交织与符号级交织有什么区别？各有何优缺点？

**扩展思考**：
7. Polar码和Turbo码的优缺点各是什么？在什么场景下你会选择其中一种？

### 9.3 延伸阅读

- Arikan, E. (2009). "Channel Polarization: A Method for Constructing Capacity-Achieving Codes"
- Berrou, C., Glavieux, A., & Thitimajshima, P. (1993). "Near Shannon Limit Error-Correcting Coding and Decoding"
- 3GPP TS 38.212: "Multiplexing and channel coding"